# 03 · Adaptive policy (≈ 12 min)
**Challenge 4: design an adaptive policy** that decides to **answer**, **verify** (run an extra, costlier check) or **abstain**.

Key ideas
* **Coverage** = fraction of questions the system answers. **Accuracy** is measured on answered questions only.
* Abstaining more always raises accuracy. So a fair test compares policies **at the same coverage**.

In [ ]:
import sys, pathlib, warnings
from collections import Counter
REPO = "pycones2026-uncertainty-workshop"   # <- must match your GitHub repo name (used on Colab only)
if "google.colab" in sys.modules:
    ROOT = pathlib.Path("/content") / REPO
    if not ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone", f"https://github.com/Shafiya0101/{REPO}.git", str(ROOT)], check=True)
else:
    ROOT = pathlib.Path.cwd()
    if ROOT.name in ("notebooks", "solutions"):
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from uncertainty_lib import *
from uncertainty_lib import checks
from uncertainty_lib.plotting import set_style, plot_region_grid, plot_risk_coverage
set_style()
pd.set_option("display.max_colwidth", 70)

In [ ]:
df = add_regions(add_signals(load_setting("mistral_triviaqa")))
checks.banner(df)
y = df.label_containment.to_numpy()
print(f"Answer everything: accuracy = {y.mean():.3f}, coverage = 1.00")

## Step 1 · Accuracy–coverage curves
Answer the lowest-uncertainty questions first. The two black markers are the fingerprint policies:
**Policy A** abstains on *Gap-like*; **Policy B** answers only *Confident*.

In [ ]:
maskA, maskB = (df.region != "Gap-like").to_numpy(), (df.region == "Confident").to_numpy()
pts = [(maskA.mean(), y[maskA].mean(), "Policy A", "*"), (maskB.mean(), y[maskB].mean(), "Policy B", "D")]
for c, a, name, _ in pts:
    print(f"{name}: coverage {c:.2f}, accuracy {a:.3f}")
plot_risk_coverage(df, "label_containment", ["U_sc", "U_se", "U_lp", "U_nli", "Comb3"], extra_points=pts)
plt.show()

**Look closely:** do the markers sit above the curves of the simple scores, or on them?

### 🛠 TODO 4: write the three-action rule

In [ ]:
def my_actions(score, t_answer, t_abstain):
    score = np.asarray(score, float)
    return np.where(score < t_answer, "answer", np.where(score < t_abstain, "verify", "abstain"))

In [ ]:
checks.check_actions(my_actions)

## Challenge 4 · Your policy
Pick two thresholds on the **Comb4** score (quantiles of the score: 0.5 means the median question).

*Toy cost model (an assumption, see `evaluate_actions`):* sampling costs 1.0 per question; *verify* adds 2.0 and
fixes the answer 90% of the time; *abstain* gives no answer.

**Leaderboard rule:** highest accuracy with coverage ≥ 0.60. Ties are broken by lower cost.

In [ ]:
q_answer, q_abstain = 0.45, 0.80          # <-- CHANGE THESE and re-run
t1, t2 = df.Comb4.quantile(q_answer), df.Comb4.quantile(q_abstain)
actions = my_actions(df.Comb4, t1, t2)
res = evaluate_actions(actions, y)
print(pd.Series(actions).value_counts().to_dict())
print(f"accuracy {res['accuracy']:.3f} | coverage {res['coverage']:.2f} | cost {res['cost_per_question']:.2f}")
print("Eligible for the leaderboard" if res["coverage"] >= 0.60 else "Coverage below 0.60: not eligible")

### Sweep: the trade-off at a glance
Each row = a policy that answers up to `q_answer`, verifies up to `q_abstain`, abstains above.

In [ ]:
rows = []
for qa in (0.3, 0.5, 0.7):
    for qb in (0.8, 0.9, 1.0):
        r = evaluate_actions(my_actions(df.Comb4, df.Comb4.quantile(qa), df.Comb4.quantile(qb) + (1e-9 if qb == 1.0 else 0)), y)
        rows.append({"q_answer": qa, "q_abstain": qb, **{k: round(v, 3) for k, v in r.items()}})
pd.DataFrame(rows)

## Step 2 · Does the *pattern* help, or just abstaining?
Compare Policy A with a **plain threshold on one score that answers the same fraction** (500 repetitions in the paper; 200 here).
If the 95% interval of the difference contains 0, the pattern adds nothing detectable.

In [ ]:
cmp = matched_coverage_comparison(df, policy="A", n_rep=200)
print(cmp.round(3))
print("\nAny interval entirely above 0?", bool((cmp.ci_low > 0).any()))

**Discuss (2 min):** what does it mean if a plain threshold matches the fingerprint? When would you still prefer the fingerprint? (Think: explaining *why* the system abstained.)

**Bonus:** repeat with `label_col="label_judge"`, or with `policy="B"`. Does the conclusion change?